# B1 · Đọc dữ liệu → BẢNG CHUẨN

Mục tiêu: đưa dữ liệu đề về **một bảng chung** để mọi bước sau dùng:

| cột | nghĩa |
|---|---|
| `id` | mã dòng (để ghép file nộp) |
| `split` | `train` / `test` |
| `image` | đường dẫn đầy đủ tới ảnh (trống nếu không có ảnh) |
| `text` | chữ / câu / câu hỏi (trống nếu không có chữ) |
| `label` | nhãn (chỉ train) |
| các cột khác | giữ nguyên |

Đường dẫn file: ở danh sách bên trái **chuột phải → Copy Path**, dán vào giữa hai dấu nháy (thêm `~/` hoặc `/home/...` ở đầu nếu cần; `L.doc_bang` tự thử thư mục nhà).

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())          # thư mục notebooks (chứa b2lib.py)
import numpy as np, pandas as pd
import b2lib as L
print("thư mục làm việc:", L.WORK)

## 1. ✏️ SỬA Ở ĐÂY: đường dẫn

In [ ]:
FILE_TRAIN = "du_lieu/train.csv"        # file bảng train (csv / xlsx / json / jsonl / parquet)
FILE_TEST  = "du_lieu/test.csv"
FILE_MAU   = "du_lieu/sample_submission.csv"   # file nộp mẫu của đề ("" nếu không có)
THU_MUC_ANH = "du_lieu/images"          # thư mục chứa ảnh nếu cột ảnh chỉ ghi tên file ("" nếu không có ảnh)

In [ ]:
tr = L.doc_bang(FILE_TRAIN); te = L.doc_bang(FILE_TEST)
print(tr.shape, te.shape); display(tr.head()); display(te.head())
if FILE_MAU: display(L.doc_bang(FILE_MAU).head())

Ảnh xếp theo thư mục lớp (`train/meo/1.jpg`, `train/cho/2.jpg`…) thay vì có file bảng? Dùng ô này thay cho ô trên:

In [ ]:
# tr = L.tu_thu_muc("du_lieu/train", co_nhan=True)      # ra cột image, label
# te = L.tu_thu_muc("du_lieu/test", co_nhan=False)      # ra cột image

## 2. ✏️ SỬA Ở ĐÂY: tên cột (xem bảng in ở trên)

In [ ]:
COT_ID   = "id"          # cột mã dòng (None nếu không có)
COT_ANH  = "image"       # cột tên / đường dẫn ảnh (None nếu không có ảnh)
COT_CHU  = ["text"]      # một hoặc nhiều cột chữ (danh sách); [] nếu không có
COT_NHAN = "label"       # cột cần đoán

A = L.chuan_hoa(tr, "train", COT_ID, COT_ANH, COT_CHU, COT_NHAN, THU_MUC_ANH or None)
B = L.chuan_hoa(te, "test",  COT_ID, COT_ANH, COT_CHU, None,    THU_MUC_ANH or None)
df = pd.concat([A, B], ignore_index=True)
L.xem_nhanh(df)

## 3. Kiểm tra dữ liệu tự động + (nếu cần) xử lý thêm: sửa chữ, bỏ dòng lỗi… viết vào ô dưới

In [ ]:
df = L.kiem_tra_du_lieu(df)

In [ ]:
# ví dụ: df["text"] = df["text"].str.strip().str.lower()
# ví dụ: df = df[~((df.split == "train") & df.label.isna())]

## 4. Xem vài mẫu bằng mắt

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
smp = df[(df.split == "train") & (df.image != "")].sample(min(8, ((df.split == "train") & (df.image != "")).sum()), random_state=0) if (df.image != "").any() else pd.DataFrame()
fig, ax = plt.subplots(1, max(1, len(smp)), figsize=(3 * max(1, len(smp)), 3.4))
for a, (_, r) in zip(np.atleast_1d(ax), smp.iterrows()):
    a.imshow(Image.open(r.image)); a.axis("off"); a.set_title(f"{r.label}\n{str(r.text)[:30]}", fontsize=8)
plt.show()
display(df[df.split == "train"][["id", "text", "label"]].head(10))

## 5. Lưu bảng chuẩn (các notebook sau đọc file này)

In [ ]:
L.luu_bang(df)